# Hugging Face 초간단 Colab 실습

- 목표: **복붙 → 실행 → 출력 해석**까지 빠르게 체험
- 범위: 텍스트(분류/번역/요약/NER/QA/생성/마스크/제로샷) + 임베딩 유사도 + 이미지 분류
- 주의: 첫 실행 시 모델이 다운로드되어 시간이 걸릴 수 있습니다. (이후 캐시로 빨라짐)

---

## 사용 방법
1. `0) 공통 설치` 셀 실행
2. 예제 1~10을 순서대로 실행
3. 각 예제의 **입력(text / image)**를 바꿔보고 결과 비교
4. 마지막 `연습문제`를 풀고 `정답` 셀로 확인

---

## 0) 공통 설치 (한 번만 실행)

- `transformers`: Hugging Face 핵심 라이브러리
- `accelerate`: 일부 파이프라인/모델 실행 보조
- `sentencepiece`: 번역/요약 등 일부 모델 토크나이저에 필요할 수 있음

In [6]:
#!pip -q install -U transformers accelerate sentencepiece

## (선택) 런타임 GPU 확인

- Colab: `런타임 → 런타임 유형 변경 → GPU`로 설정 가능
- 아래 셀에서 GPU가 잡혔는지 확인

In [7]:
import tensorflow as tf, torch

print("TF version:", tf.__version__)
print("TF GPU:", tf.config.list_physical_devices("GPU"))
print("Torch version:", torch.__version__)
print("Torch CUDA available:", torch.cuda.is_available())

TF version: 2.19.0
TF GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Torch version: 2.9.0+cu126
Torch CUDA available: True


---

# 예제 1) 감정분석 (Sentiment Analysis)

- 입력: 문장(리뷰)
- 출력: `label`, `score`
- 포인트: 가장 빠르게 "모델이 추론한다"는 감을 잡는 데모

In [8]:
from transformers import pipeline

clf = pipeline("sentiment-analysis")
# 기본 모델 자동 선택(환경/버전에 따라 달라질 수 있음)

texts = [
    "I love this movie!",
    "This is terrible.",
    "It is okay, not bad."
]

out = clf(texts)
for t, r in zip(texts, out):
    print(t, "->", r)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cuda:0


I love this movie! -> {'label': 'POSITIVE', 'score': 0.9998775720596313}
This is terrible. -> {'label': 'NEGATIVE', 'score': 0.9996345043182373}
It is okay, not bad. -> {'label': 'POSITIVE', 'score': 0.9997650980949402}


---

# 예제 2) 번역 (Korean → English)

- 입력: 한국어 문장
- 출력: 번역 결과 텍스트
- 포인트: `max_length`는 출력 토큰 길이 제한(너무 길면 잘릴 수 있음)

In [9]:
from transformers import pipeline

translator = pipeline("translation", model="Helsinki-NLP/opus-mt-ko-en")

text = "저는 콜랩에서 허깅페이스를 실행하고 있어요."
print(translator(text, max_length=64))

Device set to use cuda:0


[{'translation_text': "I'm running a Huggingspace at the Collab."}]


---

# 예제 3) 요약 (Summarization)

- 입력: 비교적 긴 텍스트(영어)
- 출력: 요약문
- 포인트:
  - `max_length`/`min_length`로 요약 길이 제어
  - 수업용으로 "입출력 구조"를 보는 목적

In [5]:
from transformers import pipeline

summ = pipeline("summarization")

doc = (
    "Hugging Face provides open-source tools and models for natural language processing and more. "
    "It enables quick prototyping with pipelines, supports fine-tuning on custom datasets, and "
    "offers an ecosystem including model hub, datasets, and spaces for demos."
)

print(summ(doc, max_length=40, min_length=15))

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Device set to use cuda:0


[{'summary_text': ' Hugging Face provides open-source tools and models for natural language processing and more . It enables quick prototyping with pipelines, supports fine-tuning on custom datasets, and offers an'}]


---

# 예제 4) 개체명 인식 (NER: Named Entity Recognition)

- 입력: 문장
- 출력: 엔티티(사람/조직/장소 등) 목록
- 포인트: `grouped_entities=True`로 토큰 단위 결과를 묶어서 보기 좋게 출력

In [10]:
from transformers import pipeline

ner = pipeline("ner", grouped_entities=True)
print(ner("Apple is looking at buying U.K. startup for $1 billion."))

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496 (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cuda:0


[{'entity_group': 'ORG', 'score': np.float32(0.9990897), 'word': 'Apple', 'start': 0, 'end': 5}, {'entity_group': 'LOC', 'score': np.float32(0.999718), 'word': 'U', 'start': 27, 'end': 28}, {'entity_group': 'LOC', 'score': np.float32(0.9987226), 'word': 'K', 'start': 29, 'end': 30}]


/usr/local/lib/python3.12/dist-packages/transformers/pipelines/token_classification.py:186: UserWarning: `grouped_entities` is deprecated and will be removed in version v5.0.0, defaulted to `aggregation_strategy="AggregationStrategy.SIMPLE"` instead.
  warnings.warn(


---

# 예제 5) 질의응답 (QA: Question Answering)

- 입력: 질문(question) + 문맥(context)
- 출력: 답(answer) + 점수(score) + 위치
- 포인트: 문맥 내에서 답을 "발췌"하는 형태

In [11]:
from transformers import pipeline

qa = pipeline("question-answering")

result = qa(
    question="Where do I live?",
    context="My name is Sarah and I live in London."
)
print(result)

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


{'score': 0.9736034870147705, 'start': 31, 'end': 37, 'answer': 'London'}


---

# 예제 6) 문장 생성 (Text Generation)

- 입력: 프롬프트(prompt)
- 출력: 생성된 문장
- 포인트:
  - 생성은 결과가 매번 조금씩 달라질 수 있음
  - `max_new_tokens`로 추가 생성 길이 제어

In [12]:
from transformers import pipeline

gen = pipeline("text-generation", model="gpt2")
print(gen("Today I learned that", max_new_tokens=30)[0]["generated_text"])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Today I learned that he'd been in the military, and so I knew it was something he'd wanted to do for years. He was in Vietnam, and he was


---

# 예제 7) 마스크 채우기 (Fill-Mask)

- 입력: [MASK] 토큰이 포함된 문장
- 출력: [MASK]에 들어갈 후보 단어와 점수
- 포인트: BERT 계열에서 대표적인 데모

In [13]:
from transformers import pipeline

fm = pipeline("fill-mask", model="bert-base-uncased")
out = fm("Paris is the [MASK] of France.")
for o in out:
    print(o["sequence"], "score=", round(o["score"], 4))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Device set to use cuda:0


paris is the capital of france. score= 0.9969
paris is the heart of france. score= 0.0006
paris is the center of france. score= 0.0004
paris is the centre of france. score= 0.0003
paris is the city of france. score= 0.0003


---

# 예제 8) 제로샷 분류 (Zero-shot Classification)

- 입력: 문장 + 후보 라벨 리스트
- 출력: 라벨별 점수
- 포인트:
  - 별도 학습 없이 후보 라벨로 분류해보는 데모
  - 문장/라벨을 바꿔가며 "결과가 왜 그렇게 나왔는지" 토론하기 좋음

In [14]:
from transformers import pipeline

zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

text = "오늘은 딥러닝 CNN 모델을 학습했어."
labels = ["sports", "technology", "cooking", "finance"]

print(zs(text, candidate_labels=labels))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


{'sequence': '오늘은 딥러닝 CNN 모델을 학습했어.', 'labels': ['cooking', 'technology', 'sports', 'finance'], 'scores': [0.37041088938713074, 0.25491923093795776, 0.25371429324150085, 0.12095552682876587]}


---

# 예제 9) 문장 임베딩 유사도 (Embedding Similarity)

- 입력: 문장 2~3개
- 출력: 코사인 유사도(숫자)
- 포인트:
  - 검색/추천/RAG에서 자주 쓰는 핵심 도구
  - 여기서는 가장 간단한 방식으로 `sentence-transformers` 사용

In [15]:
# !pip -q install -U sentence-transformers

In [16]:
from sentence_transformers import SentenceTransformer, util

m = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

s1 = "딥러닝 모델 배포 방법을 알려줘"
s2 = "머신러닝 서비스를 운영 환경에 배포하는 방법은?"
s3 = "오늘 점심 뭐 먹지?"

emb = m.encode([s1, s2, s3], convert_to_tensor=True)
sim12 = util.cos_sim(emb[0], emb[1]).item()
sim13 = util.cos_sim(emb[0], emb[2]).item()

print("sim(s1, s2) =", round(sim12, 4))
print("sim(s1, s3) =", round(sim13, 4))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

sim(s1, s2) = 0.203
sim(s1, s3) = 0.4881


---

# 예제 10) 이미지 분류 (Image Classification)

- 입력: 이미지 파일
- 출력: 상위 라벨(top-k)
- 포인트:
  - Colab에서 샘플 이미지를 다운로드해서 즉시 실행
  - 내 사진 업로드로도 확장 가능(Colab 파일 업로드 활용)

In [17]:
# !wget -q https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/coco_sample.png -O sample.png

In [18]:
from transformers import pipeline

imgclf = pipeline("image-classification", model="google/vit-base-patch16-224")
print(imgclf("sample.png")[:3])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Device set to use cuda:0


[{'label': 'Egyptian cat', 'score': 0.9374414682388306}, {'label': 'tabby, tabby cat', 'score': 0.03844264894723892}, {'label': 'tiger cat', 'score': 0.01441141590476036}]


---

# 연습문제

## Q1
예제 8(제로샷)에서 라벨 후보를 아래처럼 바꾸고 결과를 비교하세요.
- ["AI", "sports", "movie", "travel"]

## Q2
예제 2(번역)에서 입력 문장을 2개 추가해 번역 결과를 비교하세요.

## Q3
예제 9(임베딩)에서 s1과 의미가 유사한 문장을 하나 더 만들어 sim을 비교하세요.

---

아래 셀에 여러분의 답을 작성해보세요.

In [19]:
# Q1) 제로샷 라벨 바꿔보기
from transformers import pipeline

zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
text = "이번 프로젝트는 추천 시스템에 딥러닝을 적용하는 내용이야."
labels = ["AI", "sports", "movie", "travel"]

print(zs(text, candidate_labels=labels))

Device set to use cuda:0


{'sequence': '이번 프로젝트는 추천 시스템에 딥러닝을 적용하는 내용이야.', 'labels': ['travel', 'sports', 'movie', 'AI'], 'scores': [0.39774829149246216, 0.24494805932044983, 0.20166409015655518, 0.15563954412937164]}


In [20]:
# Q2) 번역 입력 추가
from transformers import pipeline

translator = pipeline("translation", model="Helsinki-NLP/opus-mt-ko-en")

texts = [
    "저는 오늘 파이썬으로 모델을 만들었어요.",
    "콜랩에서 GPU를 사용하면 학습이 빨라질 수 있어요."
]

for t in texts:
    print(t, "->", translator(t, max_length=64)[0]["translation_text"])

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cuda:0


저는 오늘 파이썬으로 모델을 만들었어요. -> I made a model today in Python.
콜랩에서 GPU를 사용하면 학습이 빨라질 수 있어요. -> With a GPU in a calllab, learning can get faster.


In [21]:
# Q3) 임베딩 유사도 문장 추가
from sentence_transformers import SentenceTransformer, util

m = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

s1 = "딥러닝 모델 배포 방법을 알려줘"
s_new = "학습한 인공지능 모델을 서버에 올려서 서비스하는 법이 궁금해"
s3 = "오늘 점심 뭐 먹지?"

emb = m.encode([s1, s_new, s3], convert_to_tensor=True)
print("sim(s1, s_new) =", round(util.cos_sim(emb[0], emb[1]).item(), 4))
print("sim(s1, s3)    =", round(util.cos_sim(emb[0], emb[2]).item(), 4))

sim(s1, s_new) = 0.3746
sim(s1, s3)    = 0.4881


---

# 정답/해설

- Q1: 제로샷은 라벨 후보에 따라 결과가 달라집니다. 라벨 문구 자체가 "모델이 이해하는 기준"이 되기 때문입니다.
- Q2: 번역은 문체/길이/전문용어에 따라 품질이 달라질 수 있습니다.
- Q3: 임베딩 유사도는 의미가 비슷할수록 숫자가 커지는 경향이 있습니다. 다만 모델/도메인에 따라 차이가 있습니다.

---

# 요약 표

| 번호 | 예제 | 입력 | 출력 | 수업 포인트 |
|---:|---|---|---|---|
| 1 | 감정분석 | 문장 | label/score | 가장 빠른 추론 체험 |
| 2 | 번역 | 한국어 | 영어 | seq2seq 기본 감각 |
| 3 | 요약 | 긴 글 | 요약문 | 길이 제어(max/min) |
| 4 | NER | 문장 | 엔티티 | 토큰→엔티티 추출 |
| 5 | QA | 질문+문단 | 답/점수 | 문맥 기반 발췌 |
| 6 | 생성 | 프롬프트 | 생성 텍스트 | 생성 모델 감각 |
| 7 | Fill-mask | [MASK] 문장 | 후보 단어 | 언어모델 기초 |
| 8 | 제로샷 | 문장+라벨 | 라벨 점수 | 라벨 설계의 영향 |
| 9 | 임베딩 | 문장들 | 유사도 수치 | 검색/추천/RAG 핵심 |
| 10 | 이미지분류 | 이미지 | 상위 라벨 | 비전 분류 체험 |